# Форматы, Parquet и сжатие — 30 заданий

Практика выполняется на eBay в `/data/raw/ebay`. Решений нет.

## Результаты обучения

После **Форматы и Parquet** вы должны объяснить внутренний механизм, предсказать изменения metadata/files, выбрать безопасную команду и доказать итог измерением, а не сообщением об успехе.

## Архитектурная модель

Parquet организует row groups → column chunks → pages; footer хранит schema/statistics. Projection и predicate pushdown уменьшают I/O, compression меняет CPU/size.

```text
client ── metadata RPC ──► NameNode
  │                         │ block locations
  └── data stream ──► DataNode 1 ──► DataNode 2

HiveServer2 ──► Metastore (schema/location/partitions)
      └──────► execution engine ──► HDFS files
```
NameNode не хранит содержимое файла, а Metastore не хранит строки таблицы.

## Физическая схема eBay

```text
/data/raw/ebay/
├── snapshot_dt=2026-06-24/part-....snappy.parquet
├── snapshot_dt=2026-06-25/part-....snappy.parquet
└── ...
```
Grain: наблюдение `itemid` в `snapshot_dt`. Группы колонок: карточка/цена,
иерархия категорий, продавец, география и доставка. Полная schema — в `data-catalog`.

Общий raw read-only; результаты принадлежат `/user/$HDFS_USER/hadoop_training` и личной Hive DB.

## Алгоритм исследования

1. Зафиксируйте path/URI, owner и ожидаемый объект. 2. Снимите состояние до. 3. Выполните одно изменение. 4. Проверьте exit code. 5. Измерьте namespace/files/bytes/schema/rows. 6. Повторите команду и оцените идемпотентность. 7. Сохраните evidence.

Измеряйте schema, codec, row groups, число/размер файлов и читаемость нужных columns/predicates.

## Типичные ошибки

- Путать локальный путь с HDFS URI.
- Делать вывод по `ls`, не проверяя blocks/bytes/schema.
- Использовать root или 777 вместо модели доступа.
- Создавать partition-каталог без Metastore или metadata без файлов.
- Считать replication резервной копией.
- Игнорировать малые файлы и цену NameNode metadata.

## Самопроверка

1. Какие metadata изменятся? 2. Где физически лежат bytes? 3. Сколько logical и physical bytes? 4. Кто может читать/писать? 5. Что произойдёт при повторе? 6. Какая независимая команда опровергнет вывод?

## Ментальная модель

Формат определяет стоимость чтения. Parquet хранит колонки, статистики и row groups; predicate pushdown и column pruning уменьшают I/O. Сжатие выбирают вместе с splittability и CPU.

## Подробная теория

### 1. Строка и колонка

CSV/JSON удобны на границе, но читают строку целиком. Parquet организует данные по колонкам и позволяет пропускать ненужные столбцы.

### 2. Физика Parquet

Файл состоит из row groups, column chunks и pages. Footer содержит схему и статистики; слишком мелкие row groups ухудшают throughput.

### 3. Pushdown

Фильтр может использовать min/max статистики и не читать row group. Это работает только при совместимых типах и предикатах.

### 4. Кодирование и compression

Dictionary/RLE уменьшают повторяющиеся значения, Snappy быстро распаковывается, Gzip сильнее сжимает ценой CPU. Выбор измеряют на своём workload.

### 5. Эволюция схемы

Добавление nullable-колонки обычно совместимо, изменение типа — нет. Читатель согласует схемы файлов, поэтому mixed schema требует теста.

## Стенд

NameNode `namenode:8020`, два DataNode, HiveServer2 `hiveserver2:10000`. Личные артефакты не создаются в общем read-only raw-слое.

## Как сдаётся задание

Валидатор проверяет артефакт и JSON-доказательство. В `command` запишите фактическую команду, в `observation` — измеренный результат, в `explanation` — почему он получился. Минимальная длина защищает от пустых ответов; содержательный смысл остаётся вашей ответственностью.

In [ ]:
import json, os, subprocess, tempfile

def save_evidence(module, task, command, observation, explanation):
    user=os.environ.get("HDFS_USER", os.environ.get("HADOOP_USER_NAME", "student"))
    target=f"/user/{user}/hadoop_training/evidence/{module}/task_{task:02d}.json"
    payload={"module":module,"task":task,"command":command,"observation":observation,"explanation":explanation}
    with tempfile.NamedTemporaryFile("w",encoding="utf-8",delete=False,suffix=".json") as f:
        json.dump(payload,f,ensure_ascii=False,indent=2); local=f.name
    subprocess.run(["hdfs","dfs","-mkdir","-p",target.rsplit("/",1)[0]],check=True)
    subprocess.run(["hdfs","dfs","-put","-f",local,target],check=True)
    os.unlink(local)
    print(target)

### Задание 1. row format

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_01` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `row format`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 1

### Задание 2. column format

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_02` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `column format`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 2

### Задание 3. CSV limits

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_03` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `CSV limits`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 3

### Задание 4. JSON lines

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_04` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `JSON lines`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 4

### Задание 5. Avro schema

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_05` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `Avro schema`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 5

### Задание 6. Parquet schema

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_06` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `Parquet schema`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 6

### Задание 7. row groups

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_07` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `row groups`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 7

### Задание 8. pages

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_08` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `pages`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 8

### Задание 9. column chunks

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_09` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `column chunks`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 9

### Задание 10. statistics

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_10` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `statistics`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 10

### Задание 11. predicate pushdown

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_11` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `predicate pushdown`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 11

### Задание 12. column pruning

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_12` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `column pruning`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 12

### Задание 13. dictionary encoding

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_13` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `dictionary encoding`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 13

### Задание 14. RLE

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_14` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `RLE`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 14

### Задание 15. Snappy

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_15` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `Snappy`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 15

### Задание 16. Gzip

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_16` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `Gzip`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 16

### Задание 17. splittability

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_17` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `splittability`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 17

### Задание 18. schema evolution

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_18` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `schema evolution`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 18

### Задание 19. nullable columns

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_19` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `nullable columns`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 19

### Задание 20. logical types

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_20` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `logical types`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 20

### Задание 21. partition columns

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_21` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `partition columns`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 21

### Задание 22. file metadata

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_22` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `file metadata`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 22

### Задание 23. pyarrow inspection

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_23` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `pyarrow inspection`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 23

### Задание 24. small files

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_24` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `small files`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 24

### Задание 25. target file size

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_25` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `target file size`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 25

### Задание 26. compaction

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_26` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `compaction`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 26

### Задание 27. compression tradeoff

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_27` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `compression tradeoff`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 27

### Задание 28. corrupt file

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_28` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `corrupt file`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 28

### Задание 29. schema reconciliation

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_29` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `schema reconciliation`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 29

### Задание 30. format decision

Создайте `/user/$HDFS_USER/hadoop_training/formats/task_30` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `format decision`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py formats 30

## Итог

Все 30 проверок должны возвращать PASS. Удалять чужие или raw-данные запрещено.